In [1]:
# ---------- CELL 1: IMPORTS + CONFIG ----------
import os
import re
import json
import math
import glob
import time
import pdfplumber
import pytesseract
from PIL import Image
from tqdm.notebook import tqdm
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss
import csv
import psycopg2   # optional
from openai import OpenAI
from pathlib import Path




C:\ProgramData\anaconda3\envs\training_env\Lib\site-packages\transformers\utils\hub.py:110: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
# ---------- CONFIG ----------
LM_BASE_URL = os.environ.get("LM_BASE_URL", "http://localhost:1234/v1")
LM_API_KEY = os.environ.get("LM_API_KEY", "lm-studio")
LM_MODEL = os.environ.get("LM_MODEL", "google/gemma-3-27b")

EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EMBED_DIM = 384

CHUNK_SIZE = 800
CHUNK_OVERLAP = 150
TOP_K = 6

# default generic paths (only used when you don't pass file-specific paths)
VECTOR_INDEX_FILE = "rag_index_eBook.faiss"
METADATA_FILE = "rag_metadata_eBook.json"
OUTPUT_CSV = "extracted_eBook.csv"

TEMPERATURE = 0.0
MAX_TOKENS = 1024

client = OpenAI(base_url=LM_BASE_URL, api_key=LM_API_KEY)



In [3]:
# ---------- HELPER: sanitize filename for index/meta ----------
def sanitize_basename(path):
    p = Path(path)
    base = p.stem.lower()
    base = re.sub(r'[^a-z0-9_]+', '_', base)
    base = re.sub(r'_{2,}', '_', base).strip('_')
    return base or "doc"

# ---------- CELL 2: OCR helpers ----------
def image_to_text(path, lang='eng'):
    try:
        img = Image.open(path)
        return pytesseract.image_to_string(img, lang=lang)
    except Exception as e:
        print(f"image_to_text error for {path}: {e}")
        return ""

def pdf_to_text(path, ocr_images=False, lang='eng'):
    pages_text = []
    try:
        with pdfplumber.open(path) as pdf:
            for p in pdf.pages:
                page_text = p.extract_text()
                if page_text and page_text.strip():
                    pages_text.append(page_text)
                else:
                    if ocr_images:
                        pil = p.to_image(resolution=300).original
                        pages_text.append(pytesseract.image_to_string(pil, lang=lang))
    except Exception as e:
        print(f"pdf_to_text error for {path}: {e}")
        return ""
    return "\n\n".join(pages_text)



In [4]:
# ---------- CELL 3: cleaning + chunking ----------
def clean_text(text):
    if not text:
        return ""
    t = re.sub(r'\r\n', '\n', text)
    t = re.sub(r'\n{3,}', '\n\n', t)
    t = re.sub(r'[ \t]+', ' ', t)
    return t.strip()

def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    text = clean_text(text)
    n = len(text)
    if n == 0:
        return []
    chunks = []
    start = 0
    cid = 0
    while start < n:
        end = start + chunk_size
        chunk_txt = text[start:end]
        chunks.append({"id": cid, "text": chunk_txt, "start": start, "end": min(end, n)})
        cid += 1
        if end >= n:
            break
        start = end - overlap
    return chunks



In [5]:
# ---------- CELL 4: embeddings + faiss helpers ----------
print("Loading embeddings model:", EMBED_MODEL_NAME)
embed_model = SentenceTransformer(EMBED_MODEL_NAME)

def create_faiss_index(dim=EMBED_DIM):
    return faiss.IndexFlatIP(dim)

def save_index(index, filepath):
    faiss.write_index(index, filepath)

def load_index(filepath):
    return faiss.read_index(filepath)

def embed_texts(texts):
    embs = embed_model.encode(texts, convert_to_numpy=True, show_progress_bar=False)
    faiss.normalize_L2(embs)
    return embs



Loading embeddings model: sentence-transformers/all-MiniLM-L6-v2


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
# ---------- CELL 5: ingest single file and folder (robust) ----------
def ingest_single_file(path, index_path=None, metadata_path=None, ocr_for_pdf=True):
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"{path} not found")
    base = sanitize_basename(path)
    if index_path is None:
        index_path = f"rag_index_{base}.faiss"
    if metadata_path is None:
        metadata_path = f"rag_meta_{base}.json"

    ext = p.suffix.lower()
    if ext == ".pdf":
        raw = pdf_to_text(str(p), ocr_images=ocr_for_pdf)
    elif ext in [".png", ".jpg", ".jpeg", ".tif", ".tiff"]:
        raw = image_to_text(str(p))
    elif ext == ".txt":
        raw = p.read_text(encoding="utf-8", errors="ignore")
    else:
        raise ValueError("Unsupported file type")

    raw = clean_text(raw)
    if not raw:
        raise RuntimeError("No text extracted. Try ocr_for_pdf=True for scanned PDFs.")

    chunks = chunk_text(raw)
    all_chunks = []
    metadata = []
    doc_id = base
    for c in chunks:
        uid = f"{doc_id}_c{c['id']}"
        all_chunks.append((uid, c["text"]))
        metadata.append({
            "uid": uid,
            "doc_id": doc_id,
            "source": str(p),
            "start": c["start"],
            "end": c["end"],
            "text": c["text"],
            "text_preview": c["text"][:400]
        })

    texts = [t for (_, t) in all_chunks]
    embs = embed_texts(texts)
    index = create_faiss_index(dim=embs.shape[1])
    index.add(embs)

    save_index(index, index_path)
    with open(metadata_path, "w", encoding="utf-8") as fh:
        json.dump(metadata, fh, indent=2, ensure_ascii=False)

    print(f"Ingested {len(all_chunks)} chunks from {path}")
    print(f"Saved index -> {index_path}")
    print(f"Saved metadata -> {metadata_path}")
    return index, metadata, index_path, metadata_path

def ingest_folder(folder, index_path=VECTOR_INDEX_FILE, metadata_path=METADATA_FILE, ocr_for_pdf=True):
    patterns = [os.path.join(folder, ext) for ext in ["**/*.pdf", "**/*.png", "**/*.jpg", "**/*.jpeg", "**/*.tif", "**/*.tiff", "**/*.txt"]]
    files = []
    for pat in patterns:
        files.extend(glob.glob(pat, recursive=True))

    all_chunks = []
    metadata = []
    doc_counter = 0
    for path in tqdm(sorted(set(files)), desc="Files"):
        ext = os.path.splitext(path)[1].lower()
        if ext == ".pdf":
            raw = pdf_to_text(path, ocr_images=ocr_for_pdf)
        elif ext in [".png", ".jpg", ".jpeg", ".tif", ".tiff"]:
            raw = image_to_text(path)
        elif ext == ".txt":
            with open(path, "r", encoding="utf-8", errors="ignore") as fh:
                raw = fh.read()
        else:
            raw = ""
        raw = clean_text(raw)
        if not raw:
            continue
        doc_id = f"doc_{doc_counter}"
        doc_counter += 1
        chunks = chunk_text(raw)
        for c in chunks:
            uid = f"{doc_id}_c{c['id']}"
            all_chunks.append((uid, c["text"]))
            metadata.append({"uid": uid, "doc_id": doc_id, "source": path, "start": c["start"], "end": c["end"], "text": c["text"], "text_preview": c["text"][:200]})

    if not all_chunks:
        print("No text chunks to ingest.")
        return None, None

    texts = [t for (_, t) in all_chunks]
    embs = embed_texts(texts)
    index = create_faiss_index(dim=embs.shape[1])
    index.add(embs)

    with open(metadata_path, "w", encoding="utf-8") as fh:
        json.dump(metadata, fh, indent=2, ensure_ascii=False)
    save_index(index, index_path)
    print(f"Ingested {len(all_chunks)} chunks from {len(metadata)} source chunks. Index saved to {index_path}")
    return index, metadata

import pytesseract
pytesseract.pytesseract.tesseract_cmd = r"D:\Kerjaan\Seksi_AD\Data_analytics\tesseract\tesseract.exe"

def ingest_path(path, index_dir=".", reuse=True, ocr_for_pdf=True):
    p = Path(path)
    if not p.exists():
        raise FileNotFoundError(f"{path} not found")
    if p.is_file():
        base = sanitize_basename(path)
        index_path = os.path.join(index_dir, f"rag_index_{base}.faiss")
        metadata_path = os.path.join(index_dir, f"rag_meta_{base}.json")
        if reuse and os.path.exists(index_path) and os.path.exists(metadata_path):
            print(f"Reusing index/metadata for {path}")
            idx = load_index(index_path)
            with open(metadata_path, "r", encoding="utf-8") as fh:
                md = json.load(fh)
            return idx, md, index_path, metadata_path
        return ingest_single_file(path, index_path=index_path, metadata_path=metadata_path, ocr_for_pdf=ocr_for_pdf) ## alternatif: ocr_for_pdf=ocr_for_pdf; ocr_for_pdf=True
    else:
        # folder: use generic names or let caller pass them
        return ingest_folder(path, index_path=os.path.join(index_dir, VECTOR_INDEX_FILE), metadata_path=os.path.join(index_dir, METADATA_FILE), ocr_for_pdf=ocr_for_pdf) + (os.path.join(index_dir, VECTOR_INDEX_FILE), os.path.join(index_dir, METADATA_FILE))



In [7]:
# ---------- CELL 6: retrieval helpers ----------
def load_vectorstore(index_path=VECTOR_INDEX_FILE, metadata_path=METADATA_FILE):
    if not os.path.exists(index_path) or not os.path.exists(metadata_path):
        raise FileNotFoundError("Index or metadata file not found. Run ingest first.")
    index = load_index(index_path)
    with open(metadata_path, "r", encoding="utf-8") as fh:
        metadata = json.load(fh)
    # Ensure text present
    for m in metadata:
        if 'text' not in m:
            m['text'] = m.get('text_preview', '')
    return index, metadata

def retrieve(query, index, metadata, top_k=TOP_K):
    q_emb = embed_texts([query])[0:1]
    D, I = index.search(q_emb, top_k)
    results = []
    for idx in I[0]:
        if idx < 0 or idx >= len(metadata):
            continue
        results.append(metadata[idx])
    return results



In [8]:
# ---------- CELL 7: LM call + JSON extraction (robust) ----------
def extract_json_block(text):
    if not text or "{" not in text:
        return None
    start_idx = None
    depth = 0
    in_string = False
    escape = False
    for i, ch in enumerate(text):
        if start_idx is None:
            if ch == "{":
                start_idx = i
                depth = 1
                in_string = False
                escape = False
            else:
                continue
        else:
            if escape:
                escape = False
                continue
            if ch == "\\":
                escape = True
                continue
            if ch == '"' or ch == "'":
                if not in_string:
                    in_string = ch
                elif in_string == ch:
                    in_string = False
                continue
            if in_string:
                continue
            if ch == "{":
                depth += 1
            elif ch == "}":
                depth -= 1
                if depth == 0:
                    candidate = text[start_idx:i+1]
                    try:
                        return json.loads(candidate)
                    except Exception:
                        try:
                            cand2 = re.sub(r"'", '"', candidate)
                            cand2 = re.sub(r",\s*}", "}", cand2)
                            cand2 = re.sub(r",\s*]", "]", cand2)
                            return json.loads(cand2)
                        except Exception:
                            return None
    return None



def call_gemma_extract_rag(retrieved_chunks, field_schema, model=LM_MODEL, temperature=TEMPERATURE, max_tokens=MAX_TOKENS):
    safe_chunks = []
    for r in (retrieved_chunks or []):
        if not isinstance(r, dict):
            continue
        text = r.get("text") or r.get("text_preview") or ""
        preview = r.get("text_preview") or (text[:200] if text else "")
        source = r.get("source") or r.get("doc_id") or "unknown"
        safe_chunks.append({"source": source, "text_preview": preview, "text": text})

    fields = field_schema if isinstance(field_schema, (list, tuple)) else list(field_schema.keys())
    schema_block = {
        "fields": [
            {"name": f, "description": (field_schema[f] if isinstance(field_schema, dict) else "")}
            for f in fields
        ],
        "rules": [
            "Output EXACTLY one valid JSON object and nothing else.",
            "Use null for missing fields.",
            "Dates use ISO 8601 if possible (YYYY-MM-DD or YYYY-MM-DDTHH:MM:SS).",
            "Do not hallucinate — if uncertain, set null and add a short note in `notes` field.",
        ]
    }

    header = "You are a data extraction engine. Given the retrieved document snippets below, extract the specified fields EXACTLY as a JSON object. Output JSON only.\n\n"
    header += json.dumps(schema_block, indent=2) + "\n\n"

    body = "## Retrieved snippets (most relevant first):\n\n"
    for i, r in enumerate(safe_chunks):
        body += f"--- snippet {i+1} (source: {r.get('source')}, preview: {r.get('text_preview')[:120]}) ---\n{r.get('text')}\n\n"

    field_example = ", ".join([f'"{f}": null' for f in fields])
    footer = f"\n\nOutput JSON with these keys: {{{field_example}, \"notes\": null}}\n\nDO NOT output any explanatory text.\n"

    prompt = header + body + footer

    try:
        resp = client.chat.completions.create(
            model=model,
            messages=[{"role":"user","content": prompt}],
            max_tokens=max_tokens,
            temperature=temperature
        )
    except Exception as e:
        return None, f"LM call failed: {e}"

    try:
        assistant_text = resp.choices[0].message.content
    except Exception:
        try:
            assistant_text = resp.choices[0]['message']['content']
        except Exception:
            assistant_text = str(resp)

    parsed = extract_json_block(assistant_text)
    return parsed, assistant_text

In [9]:


# ---------- CELL 8: FIELD SCHEMA + extractor (accepts in-memory index/metadata) ----------
FIELD_SCHEMA = {
    "Title": "Berisi judul dari dokumen elektronik eBook document",
    "Perihal": "poin penting terkait maksud dari dokumen eBook",
    "Tujuan": "Tujuan Dokumen eBook",
    "summary": "merupakan ringkasan (summary) terkait isi dokumen eBook",
    "Saran": "Yang harus dilakukan berdasarkan isi dokumen eBook",
    "Mata Kuliah": "Mata Kuliah terkait berdasarkan kurikulum top 10 universities di dunia",
    "Jurusan Kuliah": "Jurusan Kuliah terkait berdasarkan kurikulum top 10 universities di dunia",
    "Catatan": "Berisi informasi penting yang seharusnya ada atau melengkapi dokumen eBook",
}

def extract_from_all_documents(index=None, metadata=None, index_path=VECTOR_INDEX_FILE, metadata_path=METADATA_FILE, output_csv=OUTPUT_CSV, top_k=TOP_K, debug=False):
    # load or use in-memory
    if index is None or metadata is None:
        index, metadata = load_vectorstore(index_path, metadata_path)

    # ensure text fields exist
    for m in metadata:
        if 'text' not in m:
            m['text'] = m.get('text_preview','')

    docs = {}
    for m in metadata:
        docs.setdefault(m['doc_id'], []).append(m)

    results = []
    debug_outputs = []

    for doc_i, (doc_id, chunks) in enumerate(docs.items(), start=1):
        print(f"[{doc_i}/{len(docs)}] Processing doc_id={doc_id}, source={chunks[0].get('source')}")
        combined_text = " ".join([c.get('text','') for c in chunks])
        retrieved_chunks = retrieve(combined_text, index, metadata, top_k=top_k)
        parsed_json, raw_text = call_gemma_extract_rag(retrieved_chunks, FIELD_SCHEMA)

        if parsed_json is None:
            print(f"  [WARN] No valid JSON returned for {doc_id}.")
            parsed_json = {k: None for k in list(FIELD_SCHEMA.keys())}
            parsed_json['notes'] = "model_no_json"
            debug_outputs.append({"doc_id": doc_id, "source": chunks[0].get('source'), "raw_text": raw_text, "retrieved_preview": [r.get('text_preview') for r in retrieved_chunks]})
        else:
            parsed_json.setdefault('notes', None)

        parsed_json['id_dokumen'] = chunks[0].get('source')
        results.append(parsed_json)
        time.sleep(0.15)

    # write CSV
    fieldnames = list(FIELD_SCHEMA.keys()) + ["id_dokumen", "notes"]
    with open(output_csv, "w", newline="", encoding="utf-8") as fh:
        writer = csv.DictWriter(fh, fieldnames=fieldnames, extrasaction='ignore')
        writer.writeheader()
        for r in results:
            row = {k: r.get(k, None) for k in fieldnames}
            writer.writerow(row)
    print(f"Saved extracted results to {output_csv}")

    if debug and debug_outputs:
        with open("debug_raw_outputs.json", "w", encoding="utf-8") as fh:
            json.dump(debug_outputs, fh, indent=2, ensure_ascii=False)
        print(f"Saved debug details to debug_raw_outputs.json")

    return results



In [10]:
# ---------- CELL 9: USAGE EXAMPLES ----------
# Example A: process a single file explicitly (recommended)
# Replace "MY_FILE.pdf" with your exact path (full or relative)

from pathlib import Path

# Example: file is one directory up from notebook's cwd
file_path = Path("script_testing_chunk_code/input_dokumen_OCR") / "Chip Huyen - AI Engineering_ Building Applications with Foundation Models (2025, O'Reilly Media) - lib.pdf"
file_path = file_path.resolve()   # get absolute canonical path

# pass to your ingest function
index, metadata, idx_path, meta_path = ingest_path(str(file_path), index_dir=".", reuse=False, ocr_for_pdf=True)


#file_path = "Permintaan Masukan atas Substansi Rancangan Peta Jalan Kecerdasan Artifisial Kementerian Keuangan 20252029.pdf"

# Ingest specifically for that file (will create rag_index_<basename>.faiss and rag_meta_<basename>.json)
#index, metadata, idx_path, meta_path = ingest_path(file_path, index_dir=".", reuse=False, ocr_for_pdf=True)

# Run extractor using in-memory index+metadata (guaranteed to use the exact file)
results = extract_from_all_documents(index=index, metadata=metadata, output_csv=f"extracted_{sanitize_basename(file_path)}.csv", top_k=8, debug=True)
print(results[:2])

# Example B: process a folder (if you want)
# index, metadata = ingest_path("path/to/folder_with_pdfs", index_dir=".", reuse=True, ocr_for_pdf=True)
# results = extract_from_all_documents(index=index, metadata=metadata, debug=True)

Ingested 1664 chunks from D:\Kerjaan\Seksi_AD\Data_analytics\training_env\script_testing_chunk_code\input_dokumen_OCR\Chip Huyen - AI Engineering_ Building Applications with Foundation Models (2025, O'Reilly Media) - lib.pdf
Saved index -> .\rag_index_chip_huyen_ai_engineering_building_applications_with_foundation_models_2025_o_reilly_media_lib.faiss
Saved metadata -> .\rag_meta_chip_huyen_ai_engineering_building_applications_with_foundation_models_2025_o_reilly_media_lib.json
[1/1] Processing doc_id=chip_huyen_ai_engineering_building_applications_with_foundation_models_2025_o_reilly_media_lib, source=D:\Kerjaan\Seksi_AD\Data_analytics\training_env\script_testing_chunk_code\input_dokumen_OCR\Chip Huyen - AI Engineering_ Building Applications with Foundation Models (2025, O'Reilly Media) - lib.pdf
Saved extracted results to extracted_chip_huyen_ai_engineering_building_applications_with_foundation_models_2025_o_reilly_media_lib.csv
[{'Title': 'AI Engineering: Building Applications with F

In [29]:
!pip install pytesseract